# Interactive Scatter Plot of World Population Data
In this example, we visualize Gapminder demographic data using an interactive scatter plot. Interaction is achieved with CustonJS of the Bokeh package. We write JavaScript code to update the graphic. Interactive Bokeh plots based on JavaScript callback functions run on vscode.

In [ ]:
import math
import numpy as np
from bokeh.sampledata.gapminder import population as pop 
from bokeh.sampledata.gapminder import fertility as fert
from bokeh.transform import linear_cmap
from bokeh.palettes import Viridis256, Spectral11, Turbo256
from bokeh.models import ColumnDataSource, CustomJS, Slider, HoverTool, ColorBar
from bokeh.plotting import figure, show
from bokeh.layouts import column
from bokeh.io import output_notebook

output_notebook()
fert.dropna(inplace=True)
pop = pop.drop(pop.index.difference(fert.index))
countries = pop.index.tolist() # list of countries

pop = pop / 1000000 # population in millions 
max_fert = fert.max().max() # maximum fertility of a country between 1964 - 2013
min_fert = fert.min().min() # minimum fertility of a country between 1964 - 2013
max_pop = math.sqrt(pop.max().max()) # maximum population of a country between 1964 - 2013
min_pop = math.sqrt(pop.min().min()) # minimum population of a country between 1964 - 2013


config = {
    'min_radius': 5,
    'max_radius': 40,
    'max_fert': max_fert,
    'min_fert': min_fert,
    'max_pop': max_pop,
    'min_pop': min_pop,
    'population': list(zip(*map(pop.T.get,pop.T))),
    'fertility': list(zip(*map(fert.T.get,fert.T))),
    'countries': countries
}

mapper = linear_cmap(field_name='y', palette='Turbo256', low=min_fert, high=max_fert)   

def radii(pop, min_pop, max_pop):
    """
    Function to calculate the radii of the circles in the scatter chart
    based on the population of each country.
    """
    min_radius = 5 # minimum radius of the circles
    max_radius = 40 # maximum radius of the circles
    return min_radius + (np.sqrt(pop) - min_pop) / (max_pop - min_pop) * (max_radius - min_radius) # scale the radius to a maximum of 20 pixels

source = ColumnDataSource(data=dict(
    x = pop['1964'],
    y = fert['1964'],
    population = pop['1964'],
    country = countries,
    radius = radii(pop['1964'], min_pop, max_pop),
))

tooltips = [
    ('Country', '@country'),
    ('Population (millions)', '@population{0,0}'),
    ('Fertility Rate', '@y{0.00}'),
]

p = figure(title='Population vs Fertility Rate', x_axis_type='log', width=800, height=600)
p.xaxis.axis_label = 'Population (millions)'
p.yaxis.axis_label = 'Fertility Rate'
p.y_range.start = 0
p.y_range.end = 10

s_ = p.scatter(
    x='x',
    y='y',
    source=source,
    size='radius',
    fill_color=mapper,
    fill_alpha=0.4,
    line_color='#641e16',
    line_width=0.5,
    name='circle renderer',
)

p.add_tools(HoverTool(
    tooltips=tooltips,
    renderers=p.select(name='circle renderer'),
    mode='mouse',
    point_policy='follow_mouse',
))

p.title.text_font_size = '16pt'
p.title.align = 'center'
p.title.text_color = '#641e16'
p.title.text = 'Population vs Fertility Rate (1964)'


slider = Slider(start=1964, end=2013, value=1964, step=1, title='Year', width=500)
callback = CustomJS(args=dict(
    config=config,
    source=source,
    p=p), 
    code="""
    const year = cb_obj.value
    const index = year - 1964
    const { 
        population, 
        fertility, 
        countries,
        min_radius,
        max_radius,
        min_pop,
        max_pop,
        min_fert,
        max_fert
        } = config
            
    function rad_(pop, min_radius, max_radius, min_pop, max_pop) {
        const rad = pop.map( p => min_radius + (Math.sqrt(p) - min_pop) / (max_pop - min_pop) * (max_radius - min_radius) )
        return rad
    }
    const l_pop = population[index]
    source.data = {
        x: population[index],
        y: fertility[index],
        population: population[index],
        country: countries,
        radius: rad_(l_pop, min_radius, max_radius, min_pop, max_pop)
    }   
    
    p.title.text = `Population vs Fertility Rate (${year})`
    """)
slider.js_on_change('value', callback)



color_bar = ColorBar(color_mapper=mapper['transform'], width=8, location=(0,0), title='Fertility Rate')
p.add_layout(color_bar, 'right')

layout = column([slider, p])
show(layout)
#show(p)
#show(slider)

